# AGAR-RL: Autonomous Multi-Agent Deep Reinforcement Learning Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Albin0903/agario/blob/main/notebooks/train_colab.ipynb)

Ce notebook permet d'exécuter l'entraînement distribué par Deep Reinforcement Learning (PPO & Self-Play) pour générer des agents autonomes sur un environnement inspiré d'Agar.io, puis d'exporter le modèle au format universel ONNX (< 0.02 ms de latence CPU).

## 1. Cloner le Repository et Installer les Dépendances

In [ ]:
# 1. Cloner le dépôt GitHub
!git clone https://github.com/Albin0903/agario.git
%cd agario

# 2. Installer les dépendances requises
!pip install -q -r requirements.txt tensorboard

# 3. Vérifier le support GPU CUDA
import torch
print(f"CUDA disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU détecté : {torch.cuda.get_device_name(0)}")

## 2. Validation de l'Environnement et des Tests Unitaires (20 Tests)

In [ ]:
# Exécuter la suite complète de validation (moteur > 10 000 FPS, compliance Farama Gymnasium, self-play, ONNX)
!pytest -v

## 3. Lancer TensorBoard pour Suivre l'Entraînement en Direct

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/tensorboard

## 4. Démarrer l'Entraînement Distribué PPO & Self-Play

Le script utilise `SubprocVecEnv` pour faire tourner 16 environnements en parallèle avec 10 bots par arène et une pool d'adversaires mise à jour périodiquement.

In [ ]:
# Entraînement PPO (ajuste --total-timesteps selon la durée souhaitée, ex: 1_000_000 à 10_000_000)
!python src/training/train_colab.py \
    --n-envs 16 \
    --total-timesteps 1000000 \
    --batch-size 128 \
    --n-steps 2048 \
    --pool-interval 200000 \
    --device auto

## 5. Exporter la Politique vers ONNX et Benchmarker la Latence

In [ ]:
# Exportation du modèle entraîné vers ONNX avec signature statique (1, 84) -> (1, 3)
!python src/inference/export_onnx.py \
    --model checkpoints/ppo/ppo_final.zip \
    --output models/model.onnx

## 6. Télécharger le Modèle pour Déploiement Local (Ogar WebSocket)

In [ ]:
from google.colab import files

# Télécharger le modèle ONNX optimisé pour jouer sur serveur Ogar local
files.download('models/model.onnx')

# Télécharger également le checkpoint PyTorch Stable-Baselines3
files.download('checkpoints/ppo/ppo_final.zip')